# 05 — RAG Pipeline: Qualitative Demo on Clinical Queries

**Thesis reference:** Cap. 5, Sec. 5.4 — Módulo RAG  
**Objective:** Demonstrate the full Retrieval-Augmented Generation pipeline on 10 diverse clinical queries.

This notebook covers two ablation configurations:
- **A_base** — Base Llama 3.1 (8B) with no context (no retrieval, no fine-tuning)
- **C_base_rag** — Base Llama 3.1 (8B) + RAG retrieval from PubMed corpus

> **Note:** Configurations B (fine-tuned only) and D (fine-tuned + RAG) require Phase 3 (QLoRA fine-tuning), which is not yet complete. Those will be added in `06_evaluation.ipynb`.

The pipeline:
1. Encode the clinical query with **Bioformer-16L** (384d, mean pooling + L2 norm)
2. Retrieve **top-5** PubMed abstracts from Qdrant (`pubmed_abstracts`, cosine, min_year=2020)
3. Assemble a grounded prompt and call **Llama 3.1 (8B)** via HF Inference API
4. Display retrieved documents + LLM response with `[PMID: XXXXX]` citations

---

### Prerequisites
- `docker compose up -d` (Qdrant at localhost:6333, MLflow at localhost:5000)
- `uv sync --extra dev`
- `.env` configured (`HF_TOKEN`, `NCBI_EMAIL`, `QDRANT_HOST`)
- `data/raw/pubmed_bulk_corpus.json` present (~29K abstracts from notebook 02)

In [89]:
# Prerequisites check — validate all runtime dependencies before loading any model
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(override=True)

print(f"Python {sys.version}")

# Qdrant
from qdrant_client import QdrantClient as _QC
_qc = _QC(host=os.getenv("QDRANT_HOST", "localhost"), port=int(os.getenv("QDRANT_PORT", 6333)))
print(f"Qdrant connected: {_qc.get_collections()}")

# MLflow
import mlflow
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000"))
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

# Corpus
CORPUS_PATH = Path("../data/raw/pubmed_bulk_corpus.json")
assert CORPUS_PATH.exists(), f"Corpus not found: {CORPUS_PATH}"
print(f"Corpus file: {CORPUS_PATH} ({CORPUS_PATH.stat().st_size / 1e6:.1f} MB)")

# HF token
assert os.getenv("HF_TOKEN"), "HF_TOKEN not set — required for Llama 3.1 via HF Inference API"
print("HF_TOKEN configured")

# GPU (informational only — generator uses HF API, embedder runs locally)
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU detected — embedding will run on CPU (expect ~5 min for full corpus indexing)")

Python 3.12.12 (main, Feb 12 2026, 00:42:34) [Clang 21.1.4 ]
Qdrant connected: collections=[]
MLflow tracking URI: http://localhost:5001
Corpus file: ../data/raw/pubmed_bulk_corpus.json (84.1 MB)
HF_TOKEN configured
No GPU detected — embedding will run on CPU (expect ~5 min for full corpus indexing)


In [90]:
# Load RAG configuration from configs/rag.yaml
sys.path.insert(0, "..")
from src.config import load_config

cfg = load_config("rag")

print("RAG configuration loaded:")
for k, v in cfg.items():
    if isinstance(v, dict):
        print(f"  {k}:")
        for sub_k, sub_v in v.items():
            print(f"    {sub_k}: {sub_v}")
    else:
        print(f"  {k}: {v}")

RAG configuration loaded:
  embedding:
    model: bioformers/bioformer-16L
    batch_size: 32
  vector_store:
    provider: qdrant
    host: localhost
    port: 6333
    collection_name: pubmed_abstracts
    distance_metric: cosine
  retrieval:
    candidate_k: 80
    final_k: 6
    metadata_filters: {'min_year': 2015}
    hybrid_search: {'enabled': True, 'fusion': 'rrf'}
    reranker: {'enabled': True, 'model': 'ncbi/MedCPT-Cross-Encoder', 'batch_size': 16, 'max_length': 512}
    clinical_rerank: {'enabled': True, 'weights': {'cross_encoder': 0.6, 'mesh_overlap': 0.2, 'publication_type': 0.12, 'recency': 0.05, 'title_keyword': 0.03}, 'min_final_score': 0.35}
  generation:
    system_prompt: You are a clinical evidence synthesis assistant.

Use ONLY the PubMed documents provided as context. Each factual medical claim must
be supported by at least one PMID from the provided documents. Do not cite a PMID
unless that document directly supports the claim. Do not invent PMIDs.

If the retri

## Step 1: Index PubMed Corpus in Qdrant

The indexing cell is **idempotent**: it only runs if the `pubmed_abstracts` collection does not yet exist or has the wrong vector dimension (e.g. a stale 768d PubMedBERT collection).

When indexing runs:
1. Load `pubmed_bulk_corpus.json` (~29K abstracts)
2. Encode with **Bioformer-16L** (explicit mean pooling, L2 normalization → 384d vectors)
3. Upsert in batches of 256 into Qdrant with cosine distance

The `year` field is stored as an integer (defaulting to 0 for missing values) because Qdrant's range filter requires numeric comparison.

In [91]:
import json
import numpy as np
from tqdm.auto import tqdm
import transformers
from sentence_transformers import SentenceTransformer, models
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams,
    PointStruct, SparseVector,
)

transformers.logging.set_verbosity_error()

COLLECTION_NAME = cfg["vector_store"]["collection_name"]
DENSE_DIM = 384

qdrant = QdrantClient(
    host=cfg["vector_store"]["host"],
    port=cfg["vector_store"]["port"],
)

# Always recreate — collection schema has changed (added sparse vectors)
if qdrant.collection_exists(COLLECTION_NAME):
    print(f"Deleting existing collection '{COLLECTION_NAME}'...")
    qdrant.delete_collection(COLLECTION_NAME)

print("Loading corpus...")
with open(CORPUS_PATH) as f:
    corpus = json.load(f)
print(f"Loaded {len(corpus)} abstracts")

# --- Dense encoder (Bioformer-16L) ---
print(f"Loading dense embedding model: {cfg['embedding']['model']}")
word_model = models.Transformer(cfg["embedding"]["model"], max_seq_length=512)
pooling_model = models.Pooling(
    word_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False,
)
st_model = SentenceTransformer(modules=[word_model, pooling_model])
print(f"Dense model loaded | dim={st_model.get_sentence_embedding_dimension()}")

texts = [p.get("abstract", "") for p in corpus]

print("Encoding dense vectors...")
dense_embeddings = st_model.encode(
    texts,
    batch_size=cfg["embedding"]["batch_size"],
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)
print(f"Dense: {dense_embeddings.shape}")

# --- Sparse encoder (BM25 via FastEmbed) ---
# Index text = title + abstract for richer term coverage
index_texts = [
    f"{p.get('title', '')} {p.get('abstract', '')}".strip()
    for p in corpus
]

print("Initialising BM25 model...")
bm25_sparse = SparseTextEmbedding(model_name="Qdrant/bm25")
print("BM25 ready")

print("Encoding sparse vectors...")
sparse_embeddings = list(
    tqdm(bm25_sparse.embed(index_texts), total=len(index_texts), desc="BM25 encode")
)
print(f"Sparse: {len(sparse_embeddings)} vectors")

# --- Create collection with both vector spaces ---
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(size=DENSE_DIM, distance=Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams(),
    },
)
print(f"Collection '{COLLECTION_NAME}' created with dense + sparse vectors")

# --- Helper: parse raw MeSH field into clean term list ---
def parse_mesh_terms(raw_mesh: list[str]) -> list[str]:
    """
    Input:  ["*Anti-Bacterial Agents/therapeutic use", "Helicobacter pylori/physiology", "Humans"]
    Output: ["Anti-Bacterial Agents", "Helicobacter pylori", "Humans"]
    Strips leading asterisk (major topic marker) and qualifier after '/'.
    """
    terms = []
    for entry in raw_mesh:
        term = entry.lstrip("*").split("/")[0].strip()
        if term:
            terms.append(term)
    return terms

# --- Upsert ---
UPSERT_BATCH = 128

for batch_start in tqdm(range(0, len(corpus), UPSERT_BATCH), desc="Upserting"):
    batch_docs   = corpus[batch_start : batch_start + UPSERT_BATCH]
    batch_dense  = dense_embeddings[batch_start : batch_start + UPSERT_BATCH]
    batch_sparse = sparse_embeddings[batch_start : batch_start + UPSERT_BATCH]

    points = []
    for j, (doc, d_vec, s_vec) in enumerate(zip(batch_docs, batch_dense, batch_sparse)):
        points.append(
            PointStruct(
                id=batch_start + j,
                vector={
                    "dense": d_vec.tolist(),
                    "sparse": SparseVector(
                        indices=s_vec.indices.tolist(),
                        values=s_vec.values.tolist(),
                    ),
                },
                payload={
                    "pmid":              doc.get("pmid", ""),
                    "title":             doc.get("title", ""),
                    "abstract":          doc.get("abstract", ""),
                    "mesh_category":     doc.get("mesh_category", ""),
                    "mesh_terms":        parse_mesh_terms(doc.get("mesh", [])),
                    "publication_types": doc.get("publication_types", []),
                    "year":              int(doc.get("year") or 0),
                    "journal":           doc.get("journal", ""),
                    "authors":           doc.get("authors", ""),
                },
            )
        )
    qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

final_count = qdrant.get_collection(COLLECTION_NAME).points_count
print(f"Indexed {final_count} points into '{COLLECTION_NAME}'")
print("Payload fields: pmid, title, abstract, mesh_category, mesh_terms, "
    "publication_types, year, journal, authors")

Loading corpus...
Loaded 29422 abstracts
Loading dense embedding model: bioformers/bioformer-16L


Loading weights:   0%|          | 0/263 [00:00<?, ?it/s]

Dense model loaded | dim=384
Encoding dense vectors...


Batches:   0%|          | 0/920 [00:00<?, ?it/s]

Dense: (29422, 384)
Initialising BM25 model...
BM25 ready
Encoding sparse vectors...


BM25 encode:   0%|          | 0/29422 [00:00<?, ?it/s]

Sparse: 29422 vectors
Collection 'pubmed_abstracts' created with dense + sparse vectors


Upserting:   0%|          | 0/230 [00:00<?, ?it/s]

Indexed 29422 points into 'pubmed_abstracts'
Payload fields: pmid, title, abstract, mesh_category, mesh_terms, publication_types, year, journal, authors


## Step 2: Define RAG Components

In [92]:
# Embedder — Bioformer-16L with explicit mean pooling

class Embedder:
    def __init__(self, model_name: str, max_length: int = 512):
        self.model_name = model_name
        self.max_length = max_length
        self._model: SentenceTransformer | None = None

    def load_model(self) -> None:
        word_model = models.Transformer(self.model_name, max_seq_length=self.max_length)
        pooling_model = models.Pooling(
            word_model.get_word_embedding_dimension(),
            pooling_mode_mean_tokens=True,
            pooling_mode_cls_token=False,
            pooling_mode_max_tokens=False,
        )
        self._model = SentenceTransformer(modules=[word_model, pooling_model])

    def encode(self, texts: list[str], batch_size: int = 32) -> np.ndarray:
        if self._model is None:
            self.load_model()
        return self._model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False,
            convert_to_numpy=True,
        ).astype(np.float32)

    @property
    def embedding_dim(self) -> int:
        if self._model is None:
            self.load_model()
        return self._model.get_sentence_embedding_dimension()


embedder = Embedder(cfg["embedding"]["model"])
embedder.load_model()
print(f"Embedder ready: {cfg['embedding']['model']} | dim={embedder.embedding_dim}")

Loading weights:   0%|          | 0/263 [00:00<?, ?it/s]

Embedder ready: bioformers/bioformer-16L | dim=384


In [93]:
class SparseEmbedder:
    def __init__(self, bm25_model: SparseTextEmbedding):
        self._bm25 = bm25_model

    def encode_query(self, text: str) -> SparseVector:
        """
        Produces a sparse vector for a query string.
        Uses query_embed (not embed) — BM25 uses TF=1 for query terms,
        unlike document indexing which uses actual term frequencies.
        """
        result = list(self._bm25.query_embed(text))
        if not result:
            raise ValueError(f"BM25 returned no sparse vector for query: {text!r}")
        s = result[0]
        return SparseVector(indices=s.indices.tolist(), values=s.values.tolist())


sparse_embedder = SparseEmbedder(bm25_sparse)
print("SparseEmbedder ready")

SparseEmbedder ready


In [94]:
from qdrant_client.models import Filter, FieldCondition, Range, Prefetch, FusionQuery, Fusion

class Retriever:
    def __init__(
        self,
        dense_embedder: Embedder,
        sparse_embedder: SparseEmbedder,
        client: QdrantClient,
        collection_name: str,
    ):
        self.dense_embedder  = dense_embedder
        self.sparse_embedder = sparse_embedder
        self.client          = client
        self.collection_name = collection_name

    def search(
        self,
        query_text: str,
        candidate_k: int = 80,
        min_year: int | None = 2015,
    ) -> list[dict]:
        dense_vec  = self.dense_embedder.encode([query_text])[0].tolist()
        sparse_vec = self.sparse_embedder.encode_query(query_text)

        year_filter = (
            Filter(must=[FieldCondition(key="year", range=Range(gte=min_year))])
            if min_year is not None
            else None
        )

        results = self.client.query_points(
            collection_name=self.collection_name,
            prefetch=[
                Prefetch(
                    query=dense_vec,
                    using="dense",
                    filter=year_filter,
                    limit=candidate_k,
                ),
                Prefetch(
                    query=sparse_vec,
                    using="sparse",
                    filter=year_filter,
                    limit=candidate_k,
                ),
            ],
            query=FusionQuery(fusion=Fusion.RRF),
            limit=candidate_k,
        )

        return [
            {
                "rank":              i + 1,
                "pmid":              hit.payload.get("pmid", ""),
                "title":             hit.payload.get("title", ""),
                "abstract":          hit.payload.get("abstract", ""),
                "mesh_category":     hit.payload.get("mesh_category", ""),
                "mesh_terms":        hit.payload.get("mesh_terms", []),
                "publication_types": hit.payload.get("publication_types", []),
                "year":              hit.payload.get("year", 0),
                "journal":           hit.payload.get("journal", ""),
                "rrf_score":         round(hit.score, 4),
            }
            for i, hit in enumerate(results.points)
        ]


retriever = Retriever(
    dense_embedder=embedder,
    sparse_embedder=sparse_embedder,
    client=qdrant,
    collection_name=COLLECTION_NAME,
)
print("Retriever ready (hybrid: dense + BM25 sparse, RRF fusion)")

Retriever ready (hybrid: dense + BM25 sparse, RRF fusion)


In [95]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class MedCPTReranker:
    """
    Wraps ncbi/MedCPT-Cross-Encoder.
    Input:  query string + list of document dicts (must have 'title' and 'abstract')
    Output: same list with 'cross_encoder_score' added, sorted descending by score.

    MedCPT returns raw logits (unbounded floats). Higher = more relevant.
    Values are normalized to [0, 1] across the candidate batch before combining
    with other scoring signals.
    """
    MODEL_NAME = "ncbi/MedCPT-Cross-Encoder"

    def __init__(self, device: str | None = None, batch_size: int = 16, max_length: int = 512):
        self.device     = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.max_length = max_length
        self._tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)
        self._model     = (
            AutoModelForSequenceClassification
            .from_pretrained(self.MODEL_NAME)
            .to(self.device)
        )
        self._model.eval()
        print(f"MedCPTReranker loaded on {self.device}")

    def _raw_scores(self, query: str, doc_texts: list[str]) -> list[float]:
        scores = []
        for i in range(0, len(doc_texts), self.batch_size):
            batch = doc_texts[i : i + self.batch_size]
            pairs = [[query, t] for t in batch]
            encoded = self._tokenizer(
                pairs,
                truncation=True,
                padding=True,
                return_tensors="pt",
                max_length=self.max_length,
            ).to(self.device)
            with torch.no_grad():
                logits = self._model(**encoded).logits.squeeze(-1)
            scores.extend(logits.detach().cpu().tolist())
        return scores

    @staticmethod
    def _normalize(values: list[float]) -> list[float]:
        arr = np.array(values, dtype=float)
        lo, hi = arr.min(), arr.max()
        if hi == lo:
            return [0.5] * len(values)
        return ((arr - lo) / (hi - lo)).tolist()

    def rerank(self, query: str, docs: list[dict]) -> list[dict]:
        """
        Adds 'cross_encoder_score' (normalized float in [0,1]) to each doc dict.
        Returns the list sorted descending by cross_encoder_score.
        Does NOT modify the input list in-place.
        """
        doc_texts = [
            f"{d['title']}. {d['abstract']}"
            for d in docs
        ]
        raw    = self._raw_scores(query, doc_texts)
        normed = self._normalize(raw)
        result = [
            {**d, "cross_encoder_score": round(score, 4)}
            for d, score in zip(docs, normed)
        ]
        result.sort(key=lambda x: x["cross_encoder_score"], reverse=True)
        return result


reranker = MedCPTReranker(
    batch_size=cfg["retrieval"]["reranker"]["batch_size"],
    max_length=cfg["retrieval"]["reranker"]["max_length"],
)

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

MedCPTReranker loaded on cpu


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [96]:
class ClinicalScorer:
    """
    Computes a weighted clinical relevance score for each document.
    Does NOT require a query parser — works directly on the raw query string.

    Scoring components:
    - mesh_overlap:       fraction of query tokens found in doc's mesh_terms list
    - publication_type:   heuristic score based on study design hierarchy
    - recency:            decay function on publication year
    - title_keyword:      fraction of query tokens found in doc title tokens

    Weights are read from cfg["retrieval"]["clinical_rerank"]["weights"].
    """

    PUB_TYPE_SCORES = {
        "practice guideline": 1.0,
        "guideline":          1.0,
        "systematic review":  0.9,
        "meta-analysis":      0.9,
        "randomized controlled trial": 0.75,
        "clinical trial":     0.70,
        "review":             0.55,
        "journal article":    0.30,
    }

    def __init__(self, weights: dict, current_year: int = 2026):
        self.weights      = weights
        self.current_year = current_year

    def _pub_type_score(self, pub_types: list[str]) -> float:
        lower = {p.lower() for p in pub_types}
        for label, score in self.PUB_TYPE_SCORES.items():
            if label in lower:
                return score
        return 0.20

    def _recency_score(self, year: int) -> float:
        if not year:
            return 0.30
        age = self.current_year - int(year)
        if age <= 3:  return 1.00
        if age <= 7:  return 0.75
        if age <= 12: return 0.45
        return 0.20

    def _token_overlap(self, query_tokens: set[str], target_tokens: set[str]) -> float:
        if not query_tokens:
            return 0.0
        return len(query_tokens & target_tokens) / len(query_tokens)

    def score(self, query: str, doc: dict) -> dict:
        """
        Returns a dict with individual component scores and the weighted final_score.
        """
        query_tokens = set(query.lower().split())
        mesh_tokens  = {t.lower() for t in doc.get("mesh_terms", [])}
        title_tokens = set(doc.get("title", "").lower().split())

        mesh_overlap  = self._token_overlap(query_tokens, mesh_tokens)
        pub_score     = self._pub_type_score(doc.get("publication_types", []))
        rec_score     = self._recency_score(doc.get("year", 0))
        title_kw      = self._token_overlap(query_tokens, title_tokens)
        ce_score      = doc.get("cross_encoder_score", 0.0)

        w = self.weights
        final = (
            w["cross_encoder"]    * ce_score
            + w["mesh_overlap"]   * mesh_overlap
            + w["publication_type"] * pub_score
            + w["recency"]        * rec_score
            + w["title_keyword"]  * title_kw
        )

        return {
            "cross_encoder_score":  ce_score,
            "mesh_overlap":         round(mesh_overlap, 4),
            "publication_type":     round(pub_score, 4),
            "recency":              round(rec_score, 4),
            "title_keyword":        round(title_kw, 4),
            "final_score":          round(final, 4),
        }


clinical_scorer = ClinicalScorer(
    weights=cfg["retrieval"]["clinical_rerank"]["weights"],
)
print("ClinicalScorer ready")

ClinicalScorer ready


In [97]:
# Generator — Llama 3.1 (8B) via HF Inference API
from huggingface_hub import InferenceClient

class Generator:
    def __init__(
        self,
        model_id: str,
        system_prompt: str,
        max_tokens: int = 1024,
        temperature: float = 0.1,
        hf_token: str | None = None,
    ):
        self.model_id = model_id
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.temperature = temperature
        self._client = InferenceClient(token=hf_token)

    def build_prompt(self, query: str, docs: list[dict]) -> str:
        context_parts = []
        for doc in docs:
            context_parts.append(
                f"[{doc['rank']}] PMID: {doc['pmid']} | Year: {doc['year']} | MeSH: {doc['mesh_category']}\n"
                f"Title: {doc['title']}\n"
                f"Abstract: {doc['abstract']}"
            )
        context = "\n\n---\n\n".join(context_parts)
        return (
            f"=== RETRIEVED EVIDENCE ===\n\n{context}\n\n"
            f"=== END OF EVIDENCE ===\n\n"
            f"CLINICAL QUERY: {query}\n\n"
            f"EVIDENCE-BASED RESPONSE:"
        )

    def generate(self, prompt: str) -> str:
        response = self._client.chat_completion(
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt},
            ],
            model=self.model_id,
            max_tokens=self.max_tokens,
            temperature=self.temperature,
        )
        return response.choices[0].message.content

    def generate_base(self, query: str) -> str:
        """Config A_base: no retrieved context."""
        bare_prompt = f"CLINICAL QUERY: {query}\n\nEVIDENCE-BASED RESPONSE:"
        return self.generate(bare_prompt)


BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

generator = Generator(
    model_id=BASE_MODEL,
    system_prompt=cfg["generation"]["system_prompt"],
    max_tokens=cfg["generation"]["max_tokens"],
    temperature=cfg["generation"]["temperature"],
    hf_token=os.getenv("HF_TOKEN"),
)
print(f"Generator ready: {generator.model_id}")

Generator ready: meta-llama/Llama-3.1-8B-Instruct


In [98]:
import re


def verify_citations(answer: str, retrieved_pmids: list[str]) -> dict:
    cited      = set(re.findall(r"PMID[:\s]+(\d+)", answer))
    retrieved  = set(str(p) for p in retrieved_pmids)
    hallucinated = cited - retrieved
    return {
        "cited_pmids":        sorted(cited),
        "hallucinated_pmids": sorted(hallucinated),
        "citation_ok":        len(hallucinated) == 0,
    }

# RAGPipeline — orchestrates Retriever + Generator

class RAGPipeline:
    def __init__(
        self,
        retriever:        Retriever,
        reranker:         MedCPTReranker,
        clinical_scorer:  ClinicalScorer,
        generator:        Generator,
        candidate_k:      int   = 80,
        final_k:          int   = 6,
        min_year:         int   = 2015,
        min_final_score:  float = 0.35,
        min_relevant_docs: int  = 2,
    ):
        self.retriever         = retriever
        self.reranker          = reranker
        self.clinical_scorer   = clinical_scorer
        self.generator         = generator
        self.candidate_k       = candidate_k
        self.final_k           = final_k
        self.min_year          = min_year
        self.min_final_score   = min_final_score
        self.min_relevant_docs = min_relevant_docs

    def _should_abstain(self, scored_docs: list[dict]) -> bool:
        relevant = [d for d in scored_docs if d["final_score"] >= self.min_final_score]
        return len(relevant) < self.min_relevant_docs

    def run(self, query: str) -> dict:
        # 1. Hybrid retrieval (dense + BM25, RRF fusion)
        candidates = self.retriever.search(
            query, candidate_k=self.candidate_k, min_year=self.min_year
        )

        # 2. MedCPT cross-encoder reranking
        reranked = self.reranker.rerank(query, candidates)

        # 3. Clinical scoring (adds final_score to each doc)
        scored = []
        for doc in reranked:
            scores = self.clinical_scorer.score(query, doc)
            scored.append({**doc, **scores})
        scored.sort(key=lambda x: x["final_score"], reverse=True)

        # 4. Abstention check
        abstained = self._should_abstain(scored)
        if abstained:
            return {
                "query":     query,
                "abstained": True,
                "reason":    (
                    f"Fewer than {self.min_relevant_docs} retrieved documents scored "
                    f">= {self.min_final_score} after clinical reranking. "
                    "The retrieved evidence is insufficient or poorly matched to the query."
                ),
                "candidates_before_rerank": len(candidates),
                "top_candidates":           scored[:5],
                "response":                 None,
                "docs":                     [],
                "citation_check":           None,
            }

        # 5. Select final_k documents for generation
        final_docs = scored[: self.final_k]
        # Re-assign rank after reranking
        for i, d in enumerate(final_docs):
            d["rank"] = i + 1

        # 6. Generate
        prompt   = self.generator.build_prompt(query, final_docs)
        response = self.generator.generate(prompt)

        # 7. Citation verification
        citation_check = verify_citations(
            response, [d["pmid"] for d in final_docs]
        )

        avg_final_score = round(sum(d["final_score"] for d in final_docs) / len(final_docs), 4)

        return {
            "query":                    query,
            "abstained":                False,
            "candidates_before_rerank": len(candidates),
            "docs":                     final_docs,
            "prompt":                   prompt,
            "response":                 response,
            "avg_final_score":          avg_final_score,
            "citation_check":           citation_check,
        }


pipeline = RAGPipeline(
    retriever=retriever,
    reranker=reranker,
    clinical_scorer=clinical_scorer,
    generator=generator,
    candidate_k=cfg["retrieval"]["candidate_k"],
    final_k=cfg["retrieval"]["final_k"],
    min_year=cfg["retrieval"]["metadata_filters"]["min_year"],
    min_final_score=cfg["retrieval"]["clinical_rerank"]["min_final_score"],
)
print(
    f"RAGPipeline assembled | "
    f"candidate_k={pipeline.candidate_k} | final_k={pipeline.final_k} | "
    f"min_year={pipeline.min_year} | min_final_score={pipeline.min_final_score}"
)

RAGPipeline assembled | candidate_k=80 | final_k=6 | min_year=2015 | min_final_score=0.35


## Step 3: 9 Clinical Queries

| ID | MeSH Category | Clinical Topic |
|----|--------------|----------------|
| Q01 | Cardiovascular Diseases | Anticoagulation in atrial fibrillation + CKD |
| Q02 | Nervous System Diseases | DMTs for relapsing-remitting MS |
| Q03 | Mental Disorders | Treatment-resistant major depression |
| Q04 | Endocrine System Diseases | Glycemic targets in elderly T2DM with CVD risk |
| Q05 | Musculoskeletal Diseases | bDMARDs in RA after methotrexate failure |
| Q06 | Respiratory Tract Diseases | Severe persistent asthma — inhaler escalation |
| Q07 | Digestive System Diseases | H. pylori eradication with clarithromycin resistance |
| Q08 | Bacterial Infections and Mycoses | Carbapenem-resistant Klebsiella in ICU |
| Q09 | Urogenital Diseases | PSA screening harms/benefits in men 55-69 |

In [99]:
CLINICAL_QUERIES = [
    {
        "id": "Q01",
        "mesh_category": "Cardiovascular Diseases",
        "query": "What are the current evidence-based recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation and chronic kidney disease?",
    },
    {
        "id": "Q02",
        "mesh_category": "Nervous System Diseases",
        "query": "What disease-modifying therapies are recommended for relapsing-remitting multiple sclerosis in patients with high disease activity?",
    },
    {
        "id": "Q03",
        "mesh_category": "Mental Disorders",
        "query": "What pharmacological and non-pharmacological interventions are most effective for treatment-resistant major depressive disorder?",
    },
    {
        "id": "Q04",
        "mesh_category": "Endocrine System Diseases",
        "query": "What are the recommended glycemic targets and preferred antidiabetic agents for elderly patients with type 2 diabetes and high cardiovascular risk?",
    },
    {
        "id": "Q05",
        "mesh_category": "Musculoskeletal Diseases",
        "query": "What is the evidence for biological disease-modifying antirheumatic drugs in patients with rheumatoid arthritis who have failed methotrexate therapy?",
    },
    {
        "id": "Q06",
        "mesh_category": "Respiratory Tract Diseases",
        "query": "What are the recommended maintenance inhaler regimens for patients with severe persistent asthma uncontrolled on medium-dose inhaled corticosteroids?",
    },
    {
        "id": "Q07",
        "mesh_category": "Digestive System Diseases",
        "query": "What is the optimal therapeutic strategy for Helicobacter pylori eradication in regions with high clarithromycin resistance rates?",
    },
    {
        "id": "Q08",
        "mesh_category": "Bacterial Infections and Mycoses",
        "query": "What antimicrobial regimens are recommended for carbapenem-resistant Klebsiella pneumoniae infections in critically ill patients?",
    },
    {
        "id": "Q09",
        "mesh_category": "Urogenital Diseases",
        "query": "What is the current evidence for prostate-specific antigen screening in men aged 55-69 and what are the associated harms and benefits?",
    },
]

print(f"{len(CLINICAL_QUERIES)} queries across {len(set(q['mesh_category'] for q in CLINICAL_QUERIES))} MeSH categories")

9 queries across 9 MeSH categories


In [100]:
# Run all 9 queries through the RAG pipeline
import time
import pandas as pd

all_results = []

for q in CLINICAL_QUERIES:
    print(f"\n{'='*72}")
    print(f"[{q['id']}] {q['mesh_category']}")
    print(f"Query: {q['query']}")

    t0 = time.perf_counter()
    result = pipeline.run(q["query"])
    elapsed = round(time.perf_counter() - t0, 2)

    result["query_id"] = q["id"]
    result["mesh_category"] = q["mesh_category"]
    result["elapsed_s"] = elapsed
    all_results.append(result)

    if result["abstained"]:
        print(f"\n  ABSTAINED: {result['reason']}")
        print(f"  Candidates retrieved: {result['candidates_before_rerank']}")
        print(f"  Top candidate scores: {[d['final_score'] for d in result['top_candidates'][:3]]}")
        continue

    docs_df = pd.DataFrame(
        [
            {
                "Rank":        d["rank"],
                "PMID":        d["pmid"],
                "Year":        d["year"],
                "Pub Types":   ", ".join(d["publication_types"])[:40] if d["publication_types"] else "—",
                "RRF Score":   d["rrf_score"],
                "CE Score":    d["cross_encoder_score"],
                "Final Score": d["final_score"],
                "Title":       d["title"][:70] + ("..." if len(d["title"]) > 70 else ""),
            }
            for d in result["docs"]
        ]
    )
    display(docs_df)

    citation_ok = result["citation_check"]["citation_ok"]
    hallucinated = result["citation_check"]["hallucinated_pmids"]
    print(f"\nLLM Response (avg_final_score={result['avg_final_score']:.4f}, {elapsed}s):")
    print(f"Citations OK: {citation_ok}" + (f" | Hallucinated PMIDs: {hallucinated}" if not citation_ok else ""))
    print(result["response"])


[Q01] Cardiovascular Diseases
Query: What are the current evidence-based recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation and chronic kidney disease?


,Rank,PMID,Year,Pub Types,RRF Score,CE Score,Final Score,Title
0,1,29562325,2018,"Journal Article, Practice Guideline",0.5000,1.0000,0.7504,The 2018 European Heart Rhythm Association Pra...
1,2,26429352,2015,"Journal Article, Practice Guideline",0.2000,0.9891,0.7407,The 2014 Atrial Fibrillation Guidelines Compan...
2,3,26324838,2015,"Journal Article, Practice Guideline, Res",0.3333,0.9584,0.7270,Updated European Heart Rhythm Association Prac...
3,4,29562331,2018,"Journal Article, Practice Guideline, Rev",0.1667,0.9505,0.7207,The 2018 European Heart Rhythm Association Pra...
4,5,29147720,2018,"Journal Article, Practice Guideline, Rev",0.0667,0.7878,0.6247,Anticoagulation in atrial fibrillation : Curre...
5,6,30144419,2018,"Journal Article, Practice Guideline, Res",0.2500,0.7372,0.5911,Antithrombotic Therapy for Atrial Fibrillation...



LLM Response (avg_final_score=0.6924, 4.99s):
Citations OK: True
1. Recommendation or finding: 
- For patients with non-valvular atrial fibrillation and chronic kidney disease, non-vitamin K antagonist oral anticoagulants (NOACs) are recommended as an alternative to vitamin K antagonists (VKAs) to prevent stroke.
- NOACs should be used with caution in patients with chronic kidney disease, and their plasma levels should be monitored to avoid excessive anticoagulation.

2. Evidence basis: 
- [3] PMID: 26324838 (2015) - Updated European Heart Rhythm Association Practical Guide on the use of non-vitamin K antagonist anticoagulants in patients with non-valvular atrial fibrillation.
- [5] PMID: 29147720 (2018) - Anticoagulation in atrial fibrillation: Current evidence and guideline recommendations.
- [6] PMID: 30144419 (2018) - Antithrombotic Therapy for Atrial Fibrillation: CHEST Guideline and Expert Panel Report.

3. Uncertainty or gaps in the retrieved evidence: 
- The retrieved evidence

,Rank,PMID,Year,Pub Types,RRF Score,CE Score,Final Score,Title
0,1,40783682,2025,"Comparative Study, Journal Article, Netw",0.5000,1.0000,0.7700,Comparative effectiveness of disease-modifying...
1,2,37740822,2023,"Journal Article, Meta-Analysis, Systemat",0.2500,0.8820,0.6932,Stopping Disease-Modifying Treatments in Multi...
2,3,39737584,2024,"Journal Article, Practice Guideline",0.2020,0.8246,0.6728,Disease-modifying therapy in multiple sclerosi...
3,4,26237913,2015,"English Abstract, Journal Article, Pract",0.5417,0.8527,0.6561,[Update on Current Care Guideline: Multiple sc...
4,5,30794930,2019,"Journal Article, Practice Guideline, Res",0.3333,0.8211,0.6542,Autologous Hematopoietic Cell Transplantation ...
5,6,40326993,2025,"Journal Article, Practice Guideline",0.1564,0.7635,0.6341,Multiple Sclerosis Disease-Modifying Treatment...



LLM Response (avg_final_score=0.6801, 4.04s):
Citations OK: True
1. Disease-modifying therapies recommended for relapsing-remitting multiple sclerosis (RRMS) in patients with high disease activity include:
- Alemtuzumab (PMID: 40783682)
- Cladribine (PMID: 40783682, PMID: 40326993)
- Dimethyl fumarate (PMID: 40783682, PMID: 26237913)
- Fingolimod (PMID: 40783682, PMID: 26237913)
- Natalizumab (PMID: 40783682, PMID: 26237913)
- Ocrelizumab (PMID: 40326993)
- Ozanimod (PMID: 40783682)
- Ponesimod (PMID: 40783682)
- Teriflunomide (PMID: 40783682, PMID: 26237913)

2. Evidence basis:
- Systematic review and network meta-analysis (PMID: 40783682)
- Real-world studies and meta-analysis (PMID: 37740822)
- Clinical guidelines and recommendations (PMID: 26237913, PMID: 40326993)

3. Uncertainty or gaps in the retrieved evidence:
- Limited availability of head-to-head comparisons of disease-modifying therapies (PMID: 40783682)
- High risk of bias in re-analyses of individual patient data (PMID: 

...


LLM Response (avg_final_score=0.7456, 4.05s):
Citations OK: True
1. The optimal therapeutic strategy for Helicobacter pylori eradication in regions with high clarithromycin resistance rates involves using alternative therapies, including new molecular methods, and antibiotic stewardship, as well as considering regimens that have been shown to be effective in such settings, such as quinolone-based sequential therapy for 10-14 days, quinolone-based bismuth quadruple therapy for 10-14 days, bismuth quadruple therapy for 10-14 days, and quinolone-based triple therapy for 10-14 days (PMID: 32883715).
2. Evidence basis: Systematic review and network meta-analysis (PMID: 32883715), consensus guidelines (PMID: 40741103), and meta-analysis (PMID: 34092054).
3. Uncertainty or gaps in the retrieved evidence: The retrieved evidence suggests that alternative therapies may be effective in regions with high clarithromycin resistance rates, but more research is needed to determine the optimal therape

,Rank,PMID,Year,Pub Types,RRF Score,CE Score,Final Score,Title
0,1,33091620,2020,"Journal Article, Meta-Analysis, Systemat",0.0769,1.0000,0.7498,Polymyxin monotherapy versus polymyxin-based c...
1,2,34263631,2021,"Journal Article, Meta-Analysis, Systemat",0.3333,0.9660,0.7337,Predictors of mortality in patients with carba...
2,3,27624572,2017,"Comparative Study, Journal Article, Meta",0.1429,0.9625,0.7123,Polymyxin monotherapy or in combination agains...
3,4,38369125,2024,"Journal Article, Meta-Analysis, Systemat",0.1000,0.8919,0.6996,Monotherapy vs combination therapy in patients...
4,5,37768104,2023,"Journal Article, Practice Guideline",0.5667,0.8274,0.6750,Antibiotic Guidelines for Critically Ill Patie...
5,6,28356109,2017,"Journal Article, Meta-Analysis, Systemat",0.0714,0.8922,0.6701,Systematic review and meta-analysis of mortali...



LLM Response (avg_final_score=0.7067, 4.45s):
Citations OK: True
1. Recommended antimicrobial regimens for carbapenem-resistant Klebsiella pneumoniae (CR-KP) infections in critically ill patients include:
- Combination therapy with colistin and polymyxin B (PMID: 38369125)
- Combination therapy with tigecycline (PMID: 38369125)
- Combination therapy with carbapenem (PMID: 33091620)
- Combination therapy with aminoglycosides (PMID: 33091620)
- Three-drug combination therapy including polymyxin (PMID: 33091620)
- Amikacin as the most effective empiric antibiotic against Enterobacterales and A. baumannii (PMID: 37768104)
- Colistin and polymixin B showed high efficacy against all bacteria (PMID: 37768104)
- Meropenem as the first choice for the treatment of ventilator-associated pneumonia (VAP) (PMID: 37768104)
- Piperacillin-tazobactam + amikacin as the first choice for the treatment of healthcare-associated (HA) HA-pneumonia (PMID: 37768104)
- Meropenem/piperacillin-tazobactam + amikac

,Rank,PMID,Year,Pub Types,RRF Score,CE Score,Final Score,Title
0,1,31092338,2019,"Journal Article, Practice Guideline",0.3690,0.9909,0.7732,Structured Population-based Prostate-specific ...
1,2,29801017,2018,"Journal Article, Practice Guideline, Res",0.7000,1.0000,0.7569,Screening for Prostate Cancer: US Preventive S...
2,3,39041471,2024,"Journal Article, Practice Guideline",0.3860,0.8180,0.6641,The South African Prostate Cancer Screening Gu...
3,4,26087383,2015,"Journal Article, Practice Guideline",0.0833,0.7765,0.6262,Effect of the USPSTF Grade D Recommendation ag...
4,5,33172724,2021,"Journal Article, Practice Guideline, Sys",0.2000,0.7760,0.6248,EAU-EANM-ESTRO-ESUR-SIOG Guidelines on Prostat...
5,6,36031558,2022,"Journal Article, Practice Guideline, Res",0.1429,0.7646,0.6163,Discordant Endorsement of Prostate Cancer Biom...



LLM Response (avg_final_score=0.6769, 5.13s):
Citations OK: True
1. Recommendation or finding: 
- Prostate-specific antigen (PSA)-based screening for prostate cancer in men aged 55-69 years may prevent approximately 1.3 deaths from prostate cancer over approximately 13 years per 1000 men screened (PMID: 29801017).
- The decision to undergo periodic PSA-based screening for prostate cancer should be an individual one and should include discussion of the potential benefits and harms of screening with their clinician (PMID: 29801017).
- The USPSTF recommends against PSA-based screening for prostate cancer in men 70 years and older (PMID: 29801017).

2. Evidence basis: 
- Systematic review of randomized clinical trials (PMID: 29801017)
- Decision analysis models (PMID: 29801017)
- Observational study (PMID: 26087383)

3. Uncertainty or gaps in the retrieved evidence: 
- The retrieved evidence does not provide a clear consensus on the optimal use of PSA-based screening in men aged 55-69 yea

## Step 4: Base Model vs RAG-Augmented Comparison

We compare **config A_base** (no context) vs **config C_base_rag** (top-k retrieved PubMed abstracts) on the queries:

In [101]:
base_responses = {}

for q in CLINICAL_QUERIES:
    qid = q["id"]
    print(f"Generating base response for [{qid}]...")
    base_responses[qid] = generator.generate_base(q["query"])
    print(f"  Done ({len(base_responses[qid])} chars)")

Generating base response for [Q01]...
  Done (767 chars)
Generating base response for [Q02]...
  Done (1037 chars)
Generating base response for [Q03]...
  Done (1145 chars)
Generating base response for [Q04]...
  Done (2329 chars)
Generating base response for [Q05]...
  Done (1148 chars)
Generating base response for [Q06]...
  Done (2434 chars)
Generating base response for [Q07]...
  Done (700 chars)
Generating base response for [Q08]...
  Done (1082 chars)
Generating base response for [Q09]...
  Done (2194 chars)


In [102]:
# Side-by-side comparison display
for q in CLINICAL_QUERIES:
    qid = q["id"]
    rag_result = next(r for r in all_results if r["query_id"] == qid)
    query_text = q["query"]

    print(f"\n{'='*72}")
    print(f"[{qid}] {rag_result['mesh_category']}")
    print(f"Query: {query_text}")

    print(f"\n{'─'*34} A_base (no context) {'─'*18}")
    print(base_responses[qid])

    print(
        f"\n{'─'*34} C_base_rag "
        f"(candidate_k={pipeline.candidate_k}, final_k={pipeline.final_k}, "
        f"min_year={pipeline.min_year}) {'─'*6}"
    )

    if rag_result["abstained"]:
        print(f"  ABSTAINED — {rag_result['reason']}")
        continue

    print(f"Retrieved docs avg_final_score={rag_result['avg_final_score']:.4f}:")
    for d in rag_result["docs"]:
        pub_types = ", ".join(d["publication_types"])[:35] if d["publication_types"] else "—"
        print(
            f"  [{d['rank']}] PMID:{d['pmid']} ({d['year']}) "
            f"final={d['final_score']:.4f} ce={d['cross_encoder_score']:.4f} "
            f"| {pub_types} | {d['title'][:60]}"
        )
    print()
    citation_ok = rag_result["citation_check"]["citation_ok"]
    if not citation_ok:
        print(f"  WARNING: Hallucinated PMIDs: {rag_result['citation_check']['hallucinated_pmids']}")
    print(rag_result["response"])


[Q01] Cardiovascular Diseases
Query: What are the current evidence-based recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation and chronic kidney disease?

────────────────────────────────── A_base (no context) ──────────────────
1. The current evidence-based recommendations for anticoagulation therapy in patients with non-valvular atrial fibrillation (NVAF) and chronic kidney disease (CKD) include the use of non-vitamin K antagonist oral anticoagulants (NOACs) such as apixaban, dabigatran, and rivaroxaban, as well as warfarin, with careful dose adjustment and monitoring for patients with severe CKD (PMID: 33015819).
2. Evidence basis: Clinical practice guideline, observational study
3. Uncertainty or gaps in the retrieved evidence: The optimal anticoagulation strategy for patients with NVAF and CKD remains uncertain, particularly for those with severe CKD, and further research is needed to determine the most effective and safe treatment options

In [103]:
# Log experiment to MLflow
mlflow.set_experiment("05-rag-pipeline-demo")

with mlflow.start_run(run_name="hybrid_rag_qualitative_demo_v2") as run:
    mlflow.log_param("embedding_model",  cfg["embedding"]["model"])
    mlflow.log_param("llm_model",        generator.model_id)
    mlflow.log_param("collection_name",  COLLECTION_NAME)
    mlflow.log_param("candidate_k",      pipeline.candidate_k)
    mlflow.log_param("final_k",          pipeline.final_k)
    mlflow.log_param("min_year",         pipeline.min_year)
    mlflow.log_param("min_final_score",  pipeline.min_final_score)
    mlflow.log_param("temperature",      generator.temperature)
    mlflow.log_param("max_tokens",       generator.max_tokens)
    mlflow.log_param("n_queries",        len(all_results))
    mlflow.log_param("fine_tuned",       False)
    mlflow.log_param("ablation_config",  "C_base_rag")
    mlflow.log_param("hybrid_search",    "dense+bm25_rrf")
    mlflow.log_param("reranker",         cfg["retrieval"]["reranker"]["model"])

    non_abstained = [r for r in all_results if not r["abstained"]]

    for r in all_results:
        mlflow.log_metric(f"abstained_{r['query_id']}", int(r["abstained"]))
        if not r["abstained"]:
            mlflow.log_metric(f"avg_final_score_{r['query_id']}",    r["avg_final_score"])
            mlflow.log_metric(f"top1_final_score_{r['query_id']}",   r["docs"][0]["final_score"])
            mlflow.log_metric(f"top1_ce_score_{r['query_id']}",      r["docs"][0]["cross_encoder_score"])
            mlflow.log_metric(f"citation_ok_{r['query_id']}",        int(r["citation_check"]["citation_ok"]))
            mlflow.log_metric(f"hallucinated_pmids_{r['query_id']}", len(r["citation_check"]["hallucinated_pmids"]))

    if non_abstained:
        mlflow.log_metric(
            "overall_avg_final_score",
            sum(r["avg_final_score"] for r in non_abstained) / len(non_abstained),
        )
        mlflow.log_metric(
            "overall_citation_ok_rate",
            sum(int(r["citation_check"]["citation_ok"]) for r in non_abstained) / len(non_abstained),
        )
        mlflow.log_metric("n_abstained", len(all_results) - len(non_abstained))

    results_artifact = [
        {
            "query_id":                 r["query_id"],
            "mesh_category":            r["mesh_category"],
            "query":                    r["query"],
            "abstained":                r["abstained"],
            "candidates_before_rerank": r["candidates_before_rerank"],
            "avg_final_score":          r.get("avg_final_score"),
            "docs":                     r["docs"],
            "response":                 r["response"],
            "citation_check":           r["citation_check"],
            "elapsed_s":                r["elapsed_s"],
        }
        for r in all_results
    ]
    mlflow.log_dict(results_artifact, "rag_demo_results.json")

    comparison_artifact = {
        r["query_id"]: {
            "query":               r["query"],
            "base_response":       base_responses[r["query_id"]],
            "rag_response":        r["response"],
            "rag_avg_final_score": r.get("avg_final_score"),
            "abstained":           r["abstained"],
            "citation_check":      r["citation_check"],
        }
        for r in all_results
    }
    mlflow.log_dict(comparison_artifact, "base_vs_rag_comparison.json")

    print(f"Run logged: {run.info.run_id}")
    print(f"MLflow dashboard: {mlflow.get_tracking_uri()}")

Run logged: 832b9b38059c4a5ea9b4bed91162ae19
MLflow dashboard: http://localhost:5001
🏃 View run hybrid_rag_qualitative_demo_v2 at: http://localhost:5001/#/experiments/1/runs/832b9b38059c4a5ea9b4bed91162ae19
🧪 View experiment at: http://localhost:5001/#/experiments/1


### Limitations of actual RAG-System

The RAG system retrieves many irrelevant documents.

- The RAG model performs better when it retrieves directly relevant guidelines, as in the PSA and H. pylori questions. However, in many cases it retrieves documents from unrelated areas.
- This creates a different failure mode from the base model: the citations exist, but they do not actually support the claims being made.
- Average retrieval scores are high, but they do not reliably distinguish clinical relevance. Several documents with scores around 0.80 are clinically irrelevant.

---

Improvements:

- Re-rank ([Qdrant-Guide](https://qdrant.tech/documentation/tutorials-search-engineering/reranking-hybrid-search/)) with [MedCPT](https://huggingface.co/ncbi/MedCPT-Cross-Encoder) model as *Late interaction embedding* model.
- Better system prompt: More restrictive to avoid hallucinations.